**Stream-Stream Joins**

**Topic**: Streaming | **Exercises**: 7 | **Total Time**: ~110 min

Practice joining two streaming DataFrames in Structured Streaming. Covers inner and outer semantics, the role of watermarks on both sides, and how a time-range constraint together with a watermark bounds the state Spark must keep. Also covers the diagnostic skill of recognizing unbounded state growth and adding the fix.

**Solutions**: If stuck, see solutions/stream-stream-joins-solutions.py for hints and answers.

**Key concepts:**

- A stream-stream join needs both sides to be streaming DataFrames (spark.readStream.table(...)).
- Inner joins work without watermarks, but state grows unboundedly without a time bound.
- Watermark on both sides + a time-range predicate together cap the join state.
- Outer joins (left/right) REQUIRE watermark + time bound. Unmatched-side NULL rows are emitted only AFTER the watermark passes the join window.
- On Free Edition serverless, all streams must use .trigger(availableNow=True) followed by .awaitTermination(). The sources are Delta tables (Kafka is unavailable).

**- Tables** used (created by the setup notebook):

- db_code.stream_stream_joins.orders_stream (32 rows): order_id, customer_id, amount, order_ts
- db_code.stream_stream_joins.shipments_stream (32 rows): shipment_id, order_id, carrier, shipment_ts

**Pair design** (reference table for the exercises below):

**Range	Count	Time gap (shipment_ts - order_ts)	In 5-min window**?
- ORD-001..ORD-020 paired with SHP-001..SHP-020	20	+2 min	YES
- ORD-021..ORD-025 paired with SHP-021..SHP-025	5	+10 min	NO
- ORD-026..ORD-030 (no shipment)	5	n/a	unmatched left
- ORD-200..ORD-201 (late, no shipment, push watermark)	2	n/a	unmatched left
- SHP-026..SHP-030 reference ORD-901..ORD-905 (no order)	5	n/a	unmatched right
- SHP-200..SHP-201 reference ORD-906..ORD-907 (late, no order)	2	n/a	unmatched right


**Setup complete**. Two streaming source tables and the stream_stream_joins schema are ready.

Each exercise writes to its own target table (db_code.stream_stream_joins.exN_target) and uses its own checkpoint at /Volumes/db_code/stream_stream_joins/checkpoints/exN/.

**Free Edition rules** (apply to every exercise):

- Read each side with spark.readStream.table("db_code.stream_stream_joins.<source>").
- End every query with .trigger(availableNow=True) and .awaitTermination().
- Do NOT call spark.conf.set(...) - serverless rejects it.

**Exercise 1: Inner Join with No Watermark, No Time Bound**
**Difficulty**: Easy | **Time**: ~10 min

Inner-join the two streams on order_id. No watermark, no time-range predicate. Verify that every order with a matching shipment appears in the output, regardless of how far apart their timestamps are.

Source 1 (db_code.stream_stream_joins.orders_stream): 32 rows of orders. Source 2 (db_code.stream_stream_joins.shipments_stream): 32 rows of shipments.

**Target:** Write the joined stream to db_code.stream_stream_joins.ex1_target as Delta. Columns: order_id, customer_id, amount, order_ts, shipment_id, carrier, shipment_ts.

**Expected Output: **25 rows. Every order_id that exists in BOTH streams. Specifically ORD-001..ORD-025 each appear once. ORD-026..ORD-030 (no shipment), ORD-200..ORD-201 (no shipment), ORD-901..ORD-907 (no order, only orphan shipments) do NOT appear.

**Requirements:**

- Read both tables with spark.readStream.table(...).
- Inner-join on order_id.
- Write to the target Delta table with trigger(availableNow=True) and .awaitTermination().
- Set checkpointLocation to f"{CHECKPOINT_BASE}/ex1".

**Constraints:**

Do NOT add withWatermark or a time-range predicate (those come in later exercises).
Use .trigger(availableNow=True), never trigger(once=True).

In [0]:
CATALOG="db_code"
SCHEMA="stream_stream_joins"
BASE_SCHEMA="streaming"
CHECKPOINT_BASE=f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"

In [0]:
from pyspark.sql.functions import col

orders=spark.readStream.format("delta").table(f"{CATALOG}.{SCHEMA}.orders_stream")

shipments=spark.readStream.format("delta").table(f"{CATALOG}.{SCHEMA}.shipments_stream")

joined=orders.join(shipments, on="order_id", how="inner").select("order_id","customer_id","amount","order_ts","shipment_id","carrier","shipment_ts")


joined.writeStream.format("delta")\
      .option("checkpointLocation", f"{CHECKPOINT_BASE}/ex1")\
      .trigger(availableNow=True)\
      .toTable(f"{CATALOG}.{SCHEMA}.ex1_target")\
      .awaitTermination()

          



In [0]:
%sql 
select * from db_code.stream_stream_joins.ex1_target order by order_id asc

In [0]:
# Validate Exercise 1
result = spark.table(f"{CATALOG}.{SCHEMA}.ex1_target")

assert result.count() == 25, f"Expected 25 joined rows, got {result.count()}"
assert "shipment_id" in result.columns, "Output must include shipment_id from the right side"
assert "order_ts" in result.columns, "Output must include order_ts from the left side"

# Specific in-window pair (ORD-001 paired with SHP-001)
ord1 = result.filter("order_id = 'ORD-001'").collect()
assert len(ord1) == 1, f"Expected 1 row for ORD-001, got {len(ord1)}"
assert ord1[0]["shipment_id"] == "SHP-001", f"ORD-001 should pair with SHP-001, got {ord1[0]['shipment_id']}"

# Specific out-of-window pair (still matched here because no time bound)
ord21 = result.filter("order_id = 'ORD-021'").collect()
assert len(ord21) == 1, "ORD-021 should be matched (no time bound in this exercise)"
assert ord21[0]["shipment_id"] == "SHP-021", "ORD-021 should pair with SHP-021"

# Unmatched orders must not appear
assert result.filter("order_id = 'ORD-026'").count() == 0, "ORD-026 has no shipment, should not appear"
assert result.filter("order_id = 'ORD-200'").count() == 0, "ORD-200 has no shipment, should not appear"
# Orphan shipments must not appear
assert result.filter("order_id = 'ORD-901'").count() == 0, "ORD-901 only exists in shipments, should not appear"

print("Exercise 1 passed!")

**Exercise 2: Inner Join with Watermarks on Both Sides**

**Difficulty**: Medium | **Time**: ~15 min

Add withWatermark on both sides, but DO NOT yet add a time-range predicate. For inner joins, watermarks alone do not change which rows match - they only declare a notion of event-time progress. Verify the output matches Exercise 1.

The point of this exercise: watermark by itself is necessary but not sufficient to bound stream-stream join state. Without a time-range predicate, state can still grow unboundedly because Spark cannot know how late a match might arrive.

**Source 1 / Source 2: same as Exercise 1.**

**Target:** db_code.stream_stream_joins.ex2_target.

**Expected Output:** 25 rows. Same set as Exercise 1.

**Requirements:**

- Apply .withWatermark("order_ts", "1 minute") to the orders stream.
- Apply .withWatermark("shipment_ts", "1 minute") to the shipments stream.
- Inner-join on order_id. No additional time-range predicate.
- Write to the target with trigger(availableNow=True) and .awaitTermination().
- Use checkpointLocation = f"{CHECKPOINT_BASE}/ex2".

**Constraints:**

- Watermark must be applied BEFORE the join (not after).
- Delay string follows the "<value> <unit>" format (e.g., "1 minute", "10 minutes").

In [0]:
orders=spark.readStream.format("delta").table(f"{CATALOG}.{SCHEMA}.orders_stream").withWatermark("order_ts","1 minute")

shipments=spark.readStream.format("delta").table(f"{CATALOG}.{SCHEMA}.shipments_stream").withWatermark("shipment_ts","1 minute")


joined=orders.join(shipments,on="order_id",how="inner").select("order_id","customer_id","amount","order_ts","shipment_id","carrier","shipment_ts")

joined.writeStream.format("delta").option("checkpointLocation", f"{CHECKPOINT_BASE}/ex2")\
    .trigger(availableNow=True)\
    .toTable(f"{CATALOG}.{SCHEMA}.ex2_target")\
    .awaitTermination()

In [0]:
%sql 
select * from db_code.stream_stream_joins.ex2_target;

In [0]:
# Validate Exercise 2
result = spark.table(f"{CATALOG}.{SCHEMA}.ex2_target")

assert result.count() == 25, f"Expected 25 joined rows, got {result.count()}"

# In-window pair still matched
ord5 = result.filter("order_id = 'ORD-005'").collect()
assert len(ord5) == 1, "ORD-005 should be matched"
assert ord5[0]["shipment_id"] == "SHP-005", "ORD-005 should pair with SHP-005"

# Out-of-window pair still matched (no time-range predicate)
assert result.filter("order_id = 'ORD-022'").count() == 1, "ORD-022 should match SHP-022 (no time bound)"

# Unmatched and orphan rows still absent
assert result.filter("order_id = 'ORD-030'").count() == 0, "ORD-030 has no shipment"
assert result.filter("order_id = 'ORD-905'").count() == 0, "ORD-905 only exists in shipments"

print("Exercise 2 passed!")

In [0]:
# EXERCISE_KEY: stream_stream_ex3
from pyspark.sql.functions import expr

orders_wm = (
    spark.readStream.table(f"{CATALOG}.{SCHEMA}.orders_stream")
    .withWatermark("order_ts", "1 minute")
)
shipments_wm = (
    spark.readStream.table(f"{CATALOG}.{SCHEMA}.shipments_stream")
    .withWatermark("shipment_ts", "1 minute")
)

join_cond = (
    (orders_wm.order_id == shipments_wm.order_id)
    & (shipments_wm.shipment_ts >= orders_wm.order_ts)
    & (shipments_wm.shipment_ts <= orders_wm.order_ts + expr("INTERVAL 5 MINUTES"))
)

joined = orders_wm.join(shipments_wm, on=join_cond, how="inner").select(
    orders_wm.order_id,
    orders_wm.customer_id,
    orders_wm.amount,
    orders_wm.order_ts,
    shipments_wm.shipment_id,
    shipments_wm.carrier,
    shipments_wm.shipment_ts,
)

(joined.writeStream
    .format("delta")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/ex3")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.{SCHEMA}.ex3_target")
    .awaitTermination()
)

In [0]:
%sql 

select * from db_code.stream_stream_joins.ex3_target

In [0]:
# EXERCISE_KEY: stream_stream_ex4
from pyspark.sql.functions import expr

orders_wm = (
    spark.readStream.table(f"{CATALOG}.{SCHEMA}.orders_stream")
    .withWatermark("order_ts", "1 minute")
)
shipments_wm = (
    spark.readStream.table(f"{CATALOG}.{SCHEMA}.shipments_stream")
    .withWatermark("shipment_ts", "1 minute")
)

join_cond = (
    (orders_wm.order_id == shipments_wm.order_id)
    & (shipments_wm.shipment_ts >= orders_wm.order_ts)
    & (shipments_wm.shipment_ts <= orders_wm.order_ts + expr("INTERVAL 5 MINUTES"))
)

joined = orders_wm.join(shipments_wm, on=join_cond, how="leftOuter").select(
    orders_wm.order_id,
    orders_wm.customer_id,
    orders_wm.amount,
    orders_wm.order_ts,
    shipments_wm.shipment_id,
    shipments_wm.carrier,
    shipments_wm.shipment_ts,
)

(joined.writeStream
    .format("delta")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/ex4")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.{SCHEMA}.ex4_target")
    .awaitTermination()
)

In [0]:
%sql 
select * from db_code.stream_stream_joins.ex4_target order by order_id asc

In [0]:
# EXERCISE_KEY: stream_stream_ex5
from pyspark.sql.functions import expr

orders_wm = (
    spark.readStream.table(f"{CATALOG}.{SCHEMA}.orders_stream")
    .withWatermark("order_ts", "1 minute")
)
shipments_wm = (
    spark.readStream.table(f"{CATALOG}.{SCHEMA}.shipments_stream")
    .withWatermark("shipment_ts", "1 minute")
)

join_cond = (
    (orders_wm.order_id == shipments_wm.order_id)
    & (shipments_wm.shipment_ts >= orders_wm.order_ts)
    & (shipments_wm.shipment_ts <= orders_wm.order_ts + expr("INTERVAL 5 MINUTES"))
)

joined = orders_wm.join(shipments_wm, on=join_cond, how="rightOuter").select(
    orders_wm.order_id,
    orders_wm.customer_id,
    orders_wm.amount,
    orders_wm.order_ts,
    shipments_wm.shipment_id,
    shipments_wm.carrier,
    shipments_wm.shipment_ts,
)

(joined.writeStream
    .format("delta")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/ex5")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.{SCHEMA}.ex5_target")
    .awaitTermination()
)

In [0]:
%sql 
select * from db_code.stream_stream_joins.ex5_target order by shipment_id asc

In [0]:
# Validate Exercise 5
from pyspark.sql.functions import col

result = spark.table(f"{CATALOG}.{SCHEMA}.ex5_target")

assert result.count() == 30, f"Expected 30 rows (20 matched + 10 unmatched-right), got {result.count()}"

# Matched rows: 20 (have non-null order_id from the left side)
matched = result.filter("order_id IS NOT NULL")
assert matched.count() == 20, f"Expected 20 matched rows, got {matched.count()}"

# Unmatched-right rows: 10 (NULL order_id, non-null shipment_id)
unmatched = result.filter("order_id IS NULL")
assert unmatched.count() == 10, f"Expected 10 unmatched-right rows, got {unmatched.count()}"

# Specific matched row
shp7 = result.filter("shipment_id = 'SHP-007'").collect()
assert len(shp7) == 1, "SHP-007 should appear once"
assert shp7[0]["order_id"] == "ORD-007", "SHP-007 should be matched to ORD-007"

# Out-of-window pair: SHP-021 - order exists but is too early relative to shipment
shp21 = result.filter("shipment_id = 'SHP-021'").collect()
assert len(shp21) == 1, "SHP-021 should appear once (unmatched-right)"
assert shp21[0]["order_id"] is None, "SHP-021 must have NULL order_id (out of window)"

# Orphan shipment: SHP-027 references ORD-902 which doesn't exist
shp27 = result.filter("shipment_id = 'SHP-027'").collect()
assert len(shp27) == 1, "SHP-027 should appear once (unmatched-right)"
assert shp27[0]["order_id"] is None, "SHP-027 must have NULL order_id (orphan)"

# Late shipment not yet emitted
assert result.filter("shipment_id = 'SHP-200'").count() == 0, \
    "SHP-200 (late) should still be in state, not yet emitted"

print("Exercise 5 passed!")

In [0]:
orders = spark.readStream.table(f"{CATALOG}.{SCHEMA}.orders_stream")
shipments = spark.readStream.table(f"{CATALOG}.{SCHEMA}.shipments_stream")

joined = orders.join(shipments, on="order_id", how="inner").select(
    "order_id",
    "customer_id",
    "amount",
    "order_ts",
    "shipment_id",
    "carrier",
    "shipment_ts",
)

(joined.writeStream
    .format("delta")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/ex6")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.{SCHEMA}.ex6_target")
    .awaitTermination()
)

In [0]:
# Validate Exercise 6
result = spark.table(f"{CATALOG}.{SCHEMA}.ex6_target")

assert result.count() == 25, f"Expected 25 matched rows, got {result.count()}"

# Same matches as Exercise 1 - including out-of-window pairs since no time bound here
assert result.filter("order_id = 'ORD-001'").count() == 1, "ORD-001 must match"
assert result.filter("order_id = 'ORD-024'").count() == 1, "ORD-024 must match (no time bound)"
assert result.filter("order_id = 'ORD-030'").count() == 0, "ORD-030 has no shipment"

# Lesson check: this run succeeded but the pattern is dangerous in production.
print("Exercise 6 passed!")
print("LESSON: this query ran to completion under availableNow, but in a long-running stream the")
print("state would grow without bound. Compare your code to Exercise 3 to see the fix.")

In [0]:
from pyspark.sql.functions import expr

orders_wm = (
    spark.readStream.table(f"{CATALOG}.{SCHEMA}.orders_stream")
    .withWatermark("order_ts", "1 minute")
)
shipments_wm = (
    spark.readStream.table(f"{CATALOG}.{SCHEMA}.shipments_stream")
    .withWatermark("shipment_ts", "1 minute")
)

join_cond = (
    (orders_wm.order_id == shipments_wm.order_id)
    & (shipments_wm.shipment_ts >= orders_wm.order_ts)
    & (shipments_wm.shipment_ts <= orders_wm.order_ts + expr("INTERVAL 5 MINUTES"))
)

joined = orders_wm.join(shipments_wm, on=join_cond, how="inner").select(
    orders_wm.order_id,
    orders_wm.customer_id,
    orders_wm.amount,
    orders_wm.order_ts,
    shipments_wm.shipment_id,
    shipments_wm.carrier,
    shipments_wm.shipment_ts,
)

(joined.writeStream
    .format("delta")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/ex7")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.{SCHEMA}.ex7_target")
    .awaitTermination()
)

In [0]:
# Validate Exercise 7
from pyspark.sql.functions import col, expr

result = spark.table(f"{CATALOG}.{SCHEMA}.ex7_target")

assert result.count() == 20, f"Expected 20 in-window rows, got {result.count()}"

# Out-of-window pairs MUST all be dropped
for oid in ["ORD-021", "ORD-022", "ORD-023", "ORD-024", "ORD-025"]:
    assert result.filter(f"order_id = '{oid}'").count() == 0, \
        f"{oid} should be dropped (its shipment is 10 min later, out of 5-min window)"

# Every surviving row must satisfy the time bound
violations = result.filter(
    expr("shipment_ts > order_ts + INTERVAL 5 MINUTES OR shipment_ts < order_ts")
).count()
assert violations == 0, f"Found {violations} rows violating the 5-minute window bound"

# Sanity: every in-window order is present
present = set(r["order_id"] for r in result.select("order_id").collect())
for i in range(1, 21):
    oid = f"ORD-{i:03d}"
    assert oid in present, f"Missing in-window order {oid}"

print("Exercise 7 passed!")